---
## 5. Resumo Executivo da Base Analítica

Validação cruzada e métricas gerais que contextualizam a apresentação executiva:
- **Volume**: pedidos, clientes únicos, vendedores e categorias
- **Logística**: tempo médio/mediano de entrega e taxa de atraso
- **Satisfação**: cobertura de avaliações, nota média e distribuição
- **Financeiro**: ticket médio, frete médio e preço médio por pedido

In [2]:
import pandas as pd
df_base = pd.read_csv("../data/processed/base_analitica_final.csv")
olist_order_payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")


In [3]:
# ============================================================
# RESUMO EXECUTIVO DA BASE ANALÍTICA
# ============================================================
# Validação cruzada e métricas gerais que serão apresentadas
# como contexto no início da apresentação executiva.
# ============================================================

print('=' * 55)
print('RESUMO EXECUTIVO — BASE ANALÍTICA OLIST')
print('=' * 55)
print(f'\n📦 VOLUME DE DADOS')
print(f'  Pedidos analisados:              {df_base["order_id"].nunique():>8,}')
print(f'  Clientes únicos:                 {df_base["customer_unique_id"].nunique():>8,}')
print(f'  Vendedores ativos:               {df_base["seller_id"].nunique():>8,}')
print(f'  Categorias de produto:           {df_base["product_category_name_english"].nunique():>8,}')
print(f'  Estados atendidos:               {df_base["customer_state"].nunique():>8,}')

print(f'\n⏱️  LOGÍSTICA')
print(f'  Tempo médio de entrega:          {df_base["tempo_entrega_dias"].mean():>7.1f} dias')
print(f'  Tempo mediano de entrega:        {df_base["tempo_entrega_dias"].median():>7.1f} dias')
print(f'  % pedidos atrasados:             {df_base["atrasado"].mean()*100:>7.1f}%')
print(f'  Pedidos atrasados (total):       {df_base["atrasado"].sum():>8,}')

print(f'\n⭐ SATISFAÇÃO')
print(f'  Pedidos com avaliação:           {df_base["review_score"].notna().sum():>8,} ({df_base["review_score"].notna().mean()*100:.1f}%)')
print(f'  Nota média geral:                {df_base["review_score"].mean():>7.2f} / 5.0')
neg = (df_base['review_score'] <= 2).sum()
pos = (df_base['review_score'] >= 4).sum()
total_rev = df_base['review_score'].notna().sum()
print(f'  % avaliações positivas (4–5):   {pos/total_rev*100:>7.1f}%')
print(f'  % avaliações negativas (1–2):   {neg/total_rev*100:>7.1f}%')

print(f'\n💰 FINANCEIRO')
print(f'  Ticket médio (produto + frete):  R$ {df_base["ticket_total"].mean():>7.2f}')
print(f'  Frete médio por pedido:          R$ {df_base["total_freight"].mean():>7.2f}')
print(f'  Preço médio por pedido:          R$ {df_base["total_price"].mean():>7.2f}')

# --- PAGAMENTOS ---
# Agrega pagamentos por order_id (pode haver múltiplos registros por pedido)
pag = olist_order_payments.merge(
    df_base[['order_id']].drop_duplicates(), on='order_id', how='inner'
)
tipo_pag = pag.groupby('payment_type')['order_id'].nunique().sort_values(ascending=False)
total_pag = tipo_pag.sum()

parcelas = (
    pag[pag['payment_type'] == 'credit_card']
    .groupby('order_id')['payment_installments']
    .max()
)

print(f'\n💳 PAGAMENTOS')
print(f'  Tipo de pagamento (por pedido):')
for tipo, qtd in tipo_pag.items():
    print(f'    {tipo:<20} {qtd:>7,}  ({qtd/total_pag*100:.1f}%)')
print(f'  Parcelamento — cartão de crédito:')
print(f'    Parcelas médias:             {parcelas.mean():>7.1f}x')
print(f'    Mediana de parcelas:         {parcelas.median():>7.0f}x')
print(f'    % à vista (1x):             {(parcelas == 1).mean()*100:>7.1f}%')
print(f'    % parcelado (2x+):          {(parcelas >= 2).mean()*100:>7.1f}%')
print(f'    % em 10x ou mais:           {(parcelas >= 10).mean()*100:>7.1f}%')

RESUMO EXECUTIVO — BASE ANALÍTICA OLIST

📦 VOLUME DE DADOS
  Pedidos analisados:                96,456
  Clientes únicos:                   93,336
  Vendedores ativos:                  2,959
  Categorias de produto:                 72
  Estados atendidos:                     27

⏱️  LOGÍSTICA
  Tempo médio de entrega:             12.1 dias
  Tempo mediano de entrega:           10.0 dias
  % pedidos atrasados:                 6.8%
  Pedidos atrasados (total):          6,520

⭐ SATISFAÇÃO
  Pedidos com avaliação:             95,811 (99.3%)
  Nota média geral:                   4.16 / 5.0
  % avaliações positivas (4–5):      78.9%
  % avaliações negativas (1–2):      12.8%

💰 FINANCEIRO
  Ticket médio (produto + frete):  R$  159.81
  Frete médio por pedido:          R$   22.78
  Preço médio por pedido:          R$  137.03

💳 PAGAMENTOS
  Tipo de pagamento (por pedido):
    credit_card           74,287  (75.3%)
    boleto                19,187  (19.5%)
    voucher                3,679  (3.